# Xuất `merged_hf` (Qwen2.5-Coder-7B + LoRA checkpoint) trên Kaggle GPU

**Mục tiêu:** `merge_and_unload()` → thư mục Hugging Face đầy đủ trong `/kaggle/working/merged_hf`, zip để tải về.

## Chuẩn bị dataset trên Kaggle
1. Trên máy local, nén **`training/checkpoint-50`** (đủ ít nhất: `adapter_model.safetensors`, `adapter_config.json`, tokenizer: `tokenizer.json`, `tokenizer_config.json`, `chat_template.jinja` nếu có).
2. Tạo **New Dataset** trên Kaggle, upload `.zip`, đặt tên ví dụ `adba-peft-checkpoint50`.
3. Trên notebook → **Add data** → chọn dataset đó.
4. (Tuỳ chọn) Dataset có thể là thư mục `checkpoint-50` không zip — chỉnh biến `ADAPTER_SEARCH` trong code.

**Lưu ý:** Xuất **`merged_hf`** nặng **~13–15 GiB**. `/kaggle/working` và disk session phải đủ; sau khi xong có thể đẩy lên HF Hub để không phụ thuộc Kaggle disk.

In [ ]:
# Cài deps (phiên Kaggle có thể đã có torch)
%pip install -q transformers accelerate peft safetensors huggingface_hub

In [1]:
import gc
import glob
import os
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── Paths ─────────────────────────────────────────────────────────────
BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
OUTPUT_DIR = Path("/kaggle/working/merged_hf")

# Tự tìm thư mục checkpoint-50 trong các input Kaggle (đổi tên dataset nếu khác)
_cands = []
for root in glob.glob("/kaggle/input/**", recursive=False):
    for p in glob.glob(str(Path(root) / "**/checkpoint-50"), recursive=True):
        if Path(p).is_dir():
            _cands.append(Path(p))
# Trường hợp zip giải nén thẳng vào root input
for root in glob.glob("/kaggle/input/**", recursive=False):
    r = Path(root)
    if (r / "adapter_model.safetensors").is_file():
        _cands.append(r)

if not _cands:
    raise RuntimeError(
        "Không thấy LoRA checkpoint. Hãy Add Dataset chứa thư mục checkpoint-50 "
        "(hoặc file adapter nằm ngay trong thư mục dataset)."
    )

ADAPTER_PATH = max(_cands, key=lambda p: len(p.parts))  # ưu tiên path đầy đủ nhất
if not (ADAPTER_PATH / "adapter_model.safetensors").is_file():
    raise FileNotFoundError(f"Không có adapter_model.safetensors tại: {ADAPTER_PATH}")

print("ADAPTER_PATH =", ADAPTER_PATH)
print("BASE_MODEL =", BASE_MODEL)
print("OUTPUT_DIR =", OUTPUT_DIR)
print("CUDA devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {props.name}  {props.total_memory // 1024 ** 3} GiB")

# HF token: Qwen public; chỉ cần nếu limit hoặc push Hub private
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN")

# Hugging Face
_from_pretrained_kw = {"trust_remote_code": True}
if HF_TOKEN:
    _from_pretrained_kw["token"] = HF_TOKEN


ADAPTER_PATH = /kaggle/input/datasets/dangvy1507/checkpoint-50
BASE_MODEL = Qwen/Qwen2.5-Coder-7B-Instruct
OUTPUT_DIR = /kaggle/working/merged_hf
CUDA devices: 2
  [0] Tesla T4  14 GiB
  [1] Tesla T4  14 GiB


In [2]:
# Load tokenizer (ưu tiên file trong adapter để khớp template)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# T4x2 (~32GiB VRAM): device_map="auto" tách layer 2 GPU. 1x T4 có thể OOM fp16 đủ weights.
dtype = torch.float16
n_gpu = torch.cuda.device_count()

load_kw = {
    **_from_pretrained_kw,
    "torch_dtype": dtype,
    "low_cpu_mem_usage": True,
}
if n_gpu >= 2:
    load_kw["device_map"] = "auto"
    print("Dùng device_map=auto (multi-GPU).")
elif n_gpu == 1:
    load_kw["device_map"] = "auto"
    print("⚠️ 1 GPU — nếu OOM thì vào ô dưới dùng 4-bit (comment / bỏ).")
else:
    raise RuntimeError("Không có GPU — merge 7B trên CPU không thực tế trong session Kaggle.")

print("Đang load base…")
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **load_kw)
print("Đang attach LoRA…")
finetuned = PeftModel.from_pretrained(base, ADAPTER_PATH, **_from_pretrained_kw)
print("Merge LoRA vào base…")
merged = finetuned.merge_and_unload()
del base, finetuned
gc.collect()
torch.cuda.empty_cache()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Đang save merged_hf vào disk (lâu vài phút)…")
merged.save_pretrained(str(OUTPUT_DIR), safe_serialization=True, max_shard_size="4GB")
tokenizer.save_pretrained(str(OUTPUT_DIR))
print("Xong:", OUTPUT_DIR)
print("Danh mục (một phần):", sorted(p.name for p in OUTPUT_DIR.iterdir())[:20])

Dùng device_map=auto (multi-GPU).
Đang load base…


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

Đang attach LoRA…


/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Merge LoRA vào base…
Đang save merged_hf vào disk (lâu vài phút)…


Writing model shards:   0%|          | 0/4 [00:00<?, ?it/s]

Xong: /kaggle/working/merged_hf
Danh mục (một phần): ['chat_template.jinja', 'config.json', 'generation_config.json', 'model-00001-of-00004.safetensors', 'model-00002-of-00004.safetensors', 'model-00003-of-00004.safetensors', 'model-00004-of-00004.safetensors', 'model.safetensors.index.json', 'tokenizer.json', 'tokenizer_config.json']


## Nếu 1 GPU bị **CUDA OOM**
- Bật **Accelerator T4×2**, hoặc
- Thay ô load base bằng **nf4 + merge** (chậm hơn một chút, tiết kiệm VRAM). Ví dụ thêm `bitsandbytes` và dùng `BitsAndBytesConfig(load_in_4bit=True)` + `bnb_4bit_compute_dtype=torch.float16` — Phiên transformers mới (`merge_and_unload` sau 4bit) có thể dùng được; nếu lỗi version, chỉnh `%pip install` lên transformers mới hơn.

In [ ]:
# Zip để tải từ /kaggle/working (~ vài GB)
import shutil

ARCHIVE = Path("/kaggle/working/merged_hf.zip")
if ARCHIVE.is_file():
    ARCHIVE.unlink()
shutil.make_archive("/kaggle/working/merged_hf", "zip", "/kaggle/working", "merged_hf")
print("Đã zip:", ARCHIVE)

## Đẩy lên Hugging Face Hub (tuỳ chọn — tránh mang file tay)
```python
!pip install -q huggingface_hub
from huggingface_hub import login, HfApi
# login("hf_...") hoặc dùng Kaggle Secrets HF_TOKEN
api = HfApi()
# api.upload_folder(repo_id="username/adba-qwen-merged", folder_path="/kaggle/working/merged_hf", …)
```

In [3]:
import os
from huggingface_hub import HfApi, create_repo

# Đổi cho đúng user/repo của bạn
REPO_ID = "dangvanvy/adba-qwen-merged"
LOCAL_DIR = "/kaggle/working/merged_hf"

TOKEN = "..."
# Token: Kaggle Secrets tên HF_TOKEN (bạn đã add)
# try:
#     from kaggle_secrets import UserSecretsClient
#     TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
# except Exception:
#     TOKEN = os.environ["HF_TOKEN"]  # hoặc dán tạm (không commit)

create_repo(repo_id=REPO_ID, repo_type="model", private=True, exist_ok=True, token=TOKEN)

api = HfApi(token=TOKEN)
api.upload_folder(
    repo_id=REPO_ID,
    repo_type="model",
    folder_path=LOCAL_DIR,
    commit_message="Merge LoRA checkpoint-50 + Qwen2.5-Coder-7B-Instruct",
)

print(f"Đã đẩy xong: https://huggingface.co/{REPO_ID}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Đã đẩy xong: https://huggingface.co/dangvanvy/adba-qwen-merged
